# PRM Rollouts Evaluation Analysis

This notebook analyzes the evaluated rollouts from our Process Reward Model experiments.

## Scientific Plotting Theme

We use `seaborn` with a clean, colorblind-friendly style suited for scientific publications.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up scientific plotting theme
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'font.family': 'serif',
    'axes.grid': True,
    'grid.alpha': 0.5,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False
})

sns.set_theme(style="whitegrid", palette="colorblind")

In [ ]:
# Load the dataset
data_path = Path('../experiments/001_500_reasoning/data/evaluated_rollouts.jsonl')
data = []
with open(data_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Flatten into step-level dataframe
rows = []
for item in data:
    q_id = item['question_id']
    r_idx = item['rollout_index']
    for step in item.get('evaluations', []):
        rows.append({
            'question_id': q_id,
            'rollout_index': r_idx,
            'step_index': step.get('step_index'),
            'score': step.get('score'),
            'analysis_len': len(step.get('analysis', ''))
        })
df_steps = pd.DataFrame(rows)
df_steps.head()

## 10 Proposed Ways of Analysis

1. **Overall Step-Level Accuracy Distribution:** Analyze the global distribution of step scores across all rollouts.
2. **Score Progression by Step Index:** Calculate average step scores grouped by step index to see if later steps are more prone to errors.
3. **Rollout Success Rate:** Calculate the percentage of rollouts that are entirely correct (no steps with a score of 0) vs. partially/entirely incorrect.
4. **First Error Position Analysis:** Identify the `step_index` where the first error occurs in a rollout to understand where models commonly derail.
5. **Question Difficulty Ranking:** Group by `question_id` to compute average scores, identifying the hardest and easiest questions.
6. **Rollout Length vs. Accuracy:** Analyze if longer rollouts (more steps) tend to have lower average step scores or success rates.
7. **Score Variance per Question:** Measure the standard deviation of rollout success rates for a given question to understand consistency.
8. **Error Cascade Analysis:** Investigate the conditional probability of step N+1 being incorrect given that step N was incorrect.
9. **Analysis Text Length Correlation:** Correlate the length of the PRM's textual `analysis` with the assigned `score` (e.g., do negative scores require longer explanations?).
10. **Terminal State Analysis:** Examine the distribution of scores specifically on the final step of each rollout, which highly correlates with overall answer correctness.